# 🚀 YOLOv8 Treinamento e Segmentação no Database

In [ ]:
from ultralytics import YOLO
import os
from PIL import Image
import cv2

In [ ]:
# Treinar YOLOv8
import torch
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.device_count())

model = YOLO('yolov8n.pt')  

model.train(
    data="path_to_data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    project="path_to_this_file", 
    name="yolo_face_train"
)


In [ ]:
# Avaliar no conjunto de teste
model.val()

## 🧠 Aplicar modelo treinado ao banco de dados

In [ ]:
import os
import shutil
import random
from ultralytics import YOLO
from PIL import Image

# 🛠 Caminhos
ACNE_1024_DIR = r"path_dataset1"
DATASET_DIR = r"path_dataset2"
RAW_DIR = r"path_to_create_raw"
CROPS_DIR = r"path_to_create_crops"
FINAL_DIR = r"path_to_create_final"
MODEL_PATH = r"path_to_yolo_face_train/weights/best.pt"

YOLO_IMGSZ = 640
FINAL_CROP_SIZE = (224, 224)
MIN_CROP_SIZE = 30
SPLIT_RATIO = {"train": 0.7, "val": 0.15, "test": 0.15}

# 🚀 Função 1: Consolida base no raw
def copy_acne1024():
    for fname in os.listdir(ACNE_1024_DIR):
        if fname.startswith("levle"):
            src = os.path.join(ACNE_1024_DIR, fname)
            dst = os.path.join(RAW_DIR, fname)
            shutil.copy2(src, dst)

def copy_dataset():
    for split in ["Train", "Validation"]:
        split_path = os.path.join(DATASET_DIR, split)
        for level_folder in os.listdir(split_path):
            level_path = os.path.join(split_path, level_folder)
            for fname in os.listdir(level_path):
                src = os.path.join(level_path, fname)
                dst_name = f"{split}_{level_folder.replace(' ', '')}_{fname}"
                dst = os.path.join(RAW_DIR, dst_name)
                shutil.copy2(src, dst)

# 🚀 Função 2: Segmenta faces e salva crops
def segment_faces():
    model = YOLO(MODEL_PATH)
    os.makedirs(CROPS_DIR, exist_ok=True)

    for fname in os.listdir(RAW_DIR):
        if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
        fpath = os.path.join(RAW_DIR, fname)
        results = model(fpath, imgsz=YOLO_IMGSZ, save=False, stream=True)
        for r in results:
            for box in r.boxes:
                cls_name = model.names[int(box.cls)].replace("-", "_")
                region_dir = os.path.join(CROPS_DIR, cls_name)
                os.makedirs(region_dir, exist_ok=True)

                x1, y1, x2, y2 = map(int, box.xyxy[0])
                w, h = x2 - x1, y2 - y1
                if w < MIN_CROP_SIZE or h < MIN_CROP_SIZE:
                    print(f"⚠ Crop pequeno ignorado: {fname}, {cls_name}, w={w}, h={h}")
                    continue

                crop = r.orig_img[y1:y2, x1:x2]
                crop_pil = Image.fromarray(crop[..., ::-1])
                crop_resized = crop_pil.resize(FINAL_CROP_SIZE).convert("RGB")
                out_path = os.path.join(region_dir, fname)
                crop_resized.save(out_path)
    print(f"✅ Segmentação e salvamento em: {CROPS_DIR}")

# 🚀 Função 3: Split por gravidade
def extract_gravity_from_name(fname):
    fname_lower = fname.lower()
    if "levle" in fname_lower:
        try:
            return int(fname_lower.split("_")[0].replace("levle", ""))
        except ValueError:
            return -1
    elif "level0" in fname_lower:
        return 0
    elif "level1" in fname_lower:
        return 1
    elif "level2" in fname_lower:
        return 2
    elif "level3" in fname_lower:
        return 3
    else:
        return -1

def split_by_gravity():
    for region in os.listdir(CROPS_DIR):
        region_path = os.path.join(CROPS_DIR, region)
        if not os.path.isdir(region_path):
            continue
        files = os.listdir(region_path)
        gravity_files = {0: [], 1: [], 2: [], 3: []}
        for f in files:
            g = extract_gravity_from_name(f)
            if g >= 0:
                gravity_files[g].append(f)
        for g, f_list in gravity_files.items():
            random.shuffle(f_list)
            n = len(f_list)
            n_train = max(1, int(n * SPLIT_RATIO["train"])) if n > 0 else 0
            n_val = max(1, int(n * SPLIT_RATIO["val"])) if n > 1 else 0
            n_test = n - n_train - n_val
            splits = {
                "train": f_list[:n_train],
                "val": f_list[n_train:n_train + n_val],
                "test": f_list[n_train + n_val:]
            }
            for split, files in splits.items():
                out_dir = os.path.join(FINAL_DIR, region, split, str(g))
                os.makedirs(out_dir, exist_ok=True)
                for f in files:
                    shutil.copy2(os.path.join(region_path, f), os.path.join(out_dir, f))
            print(f"{region} | {g} | train: {len(splits['train'])}, val: {len(splits['val'])}, test: {len(splits['test'])}")
    print(f"✅ Split final concluído em: {FINAL_DIR}")

# 🚀 Main
def main():
    os.makedirs(RAW_DIR, exist_ok=True)
    copy_acne1024()
    copy_dataset()
    print(f"✅ Dados crus consolidados em: {RAW_DIR}")

    segment_faces()
    split_by_gravity()

if __name__ == "__main__":
    main()


In [ ]:
import os
import shutil
import random
from PIL import Image, ImageOps, ImageEnhance

PROCESSED_DIR = r"path_to_crops"
FINAL_DIR = r"path_to_final"
SPLIT_RATIO = {"train": 0.7, "val": 0.15, "test": 0.15}
AUGS_PER_IMAGE = 3 

def augment_image(img):
    augmentations = []
    for _ in range(AUGS_PER_IMAGE):
        aug = img.copy()
        if random.random() < 0.5:
            aug = ImageOps.mirror(aug)
        if random.random() < 0.5:
            angle = random.randint(-15, 15)
            aug = aug.rotate(angle)
        if random.random() < 0.5:
            enhancer = ImageEnhance.Contrast(aug)
            aug = enhancer.enhance(random.uniform(0.8, 1.2))
        augmentations.append(aug)
    return augmentations

def extract_gravity_from_name(fname):
    fname_lower = fname.lower()
    if "levle" in fname_lower:
        try:
            return int(fname_lower.split("_")[0].replace("levle", ""))
        except ValueError:
            return -1
    elif "level0" in fname_lower:
        return 0
    elif "level1" in fname_lower:
        return 1
    elif "level2" in fname_lower:
        return 2
    elif "level3" in fname_lower:
        return 3
    else:
        return -1

def main():
    for region in os.listdir(PROCESSED_DIR):
        region_path = os.path.join(PROCESSED_DIR, region)
        if not os.path.isdir(region_path):
            continue
        files = os.listdir(region_path)
        gravity_files = {0: [], 1: [], 2: [], 3: []}
        for f in files:
            g = extract_gravity_from_name(f)
            if g >= 0:
                gravity_files[g].append(f)

        for g, f_list in gravity_files.items():
            random.shuffle(f_list)
            n = len(f_list)
            n_train = max(1, int(n * SPLIT_RATIO["train"])) if n > 0 else 0
            n_val = max(1, int(n * SPLIT_RATIO["val"])) if n > 1 else 0
            n_test = n - n_train - n_val

            splits = {
                "train": f_list[:n_train],
                "val": f_list[n_train:n_train + n_val],
                "test": f_list[n_train + n_val:]
            }

            if len(splits["val"]) == 0 and splits["train"]:
                splits["val"].append(splits["train"][0])
            if len(splits["test"]) == 0 and splits["train"]:
                splits["test"].append(splits["train"][-1])

            for split, files in splits.items():
                out_dir = os.path.join(FINAL_DIR, region, split, str(g))
                os.makedirs(out_dir, exist_ok=True)
                for f in files:
                    src_path = os.path.join(region_path, f)
                    shutil.copy2(src_path, os.path.join(out_dir, f))

                    if split == "train" and n < 10:
                        img = Image.open(src_path).convert("RGB")
                        aug_imgs = augment_image(img)
                        for idx, aug in enumerate(aug_imgs):
                            aug_name = f"aug_{idx}_{f}"
                            aug.save(os.path.join(out_dir, aug_name))

            print(f"{region} | {g} | train: {len(splits['train'])}, val: {len(splits['val'])}, test: {len(splits['test'])}")

    print(f"✅ Split e augmentation concluídos em: {FINAL_DIR}")

if __name__ == "__main__":
    main()


In [ ]:
import os
import random
from PIL import Image, ImageOps, ImageEnhance

FINAL_DIR = r"path_to_final"
BOOST_CLASSES = [3]  
MIN_TARGET = 50      
AUGS_PER_IMAGE = 5   

def augment_image(img):
    aug = img.copy()
    if random.random() < 0.5:
        aug = ImageOps.mirror(aug)
    if random.random() < 0.5:
        angle = random.randint(-15, 15)
        aug = aug.rotate(angle)
    if random.random() < 0.5:
        enhancer = ImageEnhance.Contrast(aug)
        aug = enhancer.enhance(random.uniform(0.8, 1.2))
    if random.random() < 0.5:
        enhancer = ImageEnhance.Brightness(aug)
        aug = enhancer.enhance(random.uniform(0.8, 1.2))
    return aug

def boost_split(split_path):
    files = [f for f in os.listdir(split_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    while len(files) < MIN_TARGET:
        if not files:
            break 
        f = random.choice(files)
        img_path = os.path.join(split_path, f)
        img = Image.open(img_path).convert("RGB")
        aug = augment_image(img)
        aug_name = f"boost_{len(files)}_{f}"
        aug.save(os.path.join(split_path, aug_name))
        files.append(aug_name)

def main():
    for region in os.listdir(FINAL_DIR):
        for split in ["train", "val", "test"]:
            for cls in BOOST_CLASSES:
                split_path = os.path.join(FINAL_DIR, region, split, str(cls))
                if os.path.isdir(split_path):
                    boost_split(split_path)
                    n_files = len([f for f in os.listdir(split_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
                    print(f"{region}/{split}/{cls}: {n_files} imagens após boost")

    print("✅ Augmentation boost concluído!")

if __name__ == "__main__":
    main()
